# Publications HAL par laboratoire (2015–2025)

Interrogation de l'API HAL (`https://api.archives-ouvertes.fr/search/`) par `structId_i` pour chaque laboratoire.

In [1]:
import requests
import pandas as pd
import time
from tqdm.notebook import tqdm

In [2]:

# ── Chargement depuis le cache (hal_publications_par_labo.csv / hal_publications_uniques.csv) ──
import os

_CACHE_PAR_LABO   = "hal_publications_par_labo.csv"
_CACHE_UNIQUES    = "hal_publications_uniques.csv"

_REQUIRED_COLS = ["laboratoire","auteurs","noms","prenoms","idhal","orcids",
                  "titre","doi","annee","type_doc","langue","editeur",
                  "journal_revue","conference","hal_id"]

def _has_orcid_data(df_):
    """Renvoie True si la colonne 'orcids' existe et contient au moins une valeur non vide."""
    if "orcids" not in df_.columns:
        return False
    return df_["orcids"].dropna().astype(str).str.strip().ne("").any()

HAL_LOADED_FROM_CACHE = False

if os.path.exists(_CACHE_PAR_LABO) and os.path.exists(_CACHE_UNIQUES):
    _df_par_labo = pd.read_csv(_CACHE_PAR_LABO, encoding="utf-8-sig", low_memory=False)
    _df_uniques  = pd.read_csv(_CACHE_UNIQUES,  encoding="utf-8-sig", low_memory=False)

    # Vérification des colonnes obligatoires
    _missing_labo   = [c for c in _REQUIRED_COLS if c not in _df_par_labo.columns]
    _missing_unique = [c for c in _REQUIRED_COLS if c not in _df_uniques.columns]

    if _missing_labo or _missing_unique:
        print("⚠️  Fichiers cache trouvés mais colonnes manquantes :")
        if _missing_labo:
            print(f"   {_CACHE_PAR_LABO} : {_missing_labo}")
        if _missing_unique:
            print(f"   {_CACHE_UNIQUES} : {_missing_unique}")
        print("→ Relancez les cellules de collecte HAL.")
    elif not _has_orcid_data(_df_par_labo):
        print("⚠️  Fichiers cache trouvés mais la colonne 'orcids' est vide.")
        print("→ Relancez les cellules de collecte HAL et d'enrichissement ORCID.")
    else:
        df        = _df_par_labo.copy()
        df_unique = _df_uniques.copy()
        # Restaurer le type entier pour l'année
        df["annee"]        = pd.to_numeric(df["annee"],        errors="coerce").astype("Int64")
        df_unique["annee"] = pd.to_numeric(df_unique["annee"], errors="coerce").astype("Int64")
        HAL_LOADED_FROM_CACHE = True
        n_orcid = df["orcids"].dropna().astype(str).str.strip().ne("").sum()
        print(f"✅ Cache chargé depuis :")
        print(f"   • {_CACHE_PAR_LABO}  → {len(df):,} lignes")
        print(f"   • {_CACHE_UNIQUES}   → {len(df_unique):,} lignes")
        print(f"   • Publications avec ORCID : {n_orcid:,}")
        print()
        print("ℹ️  Cellules à ignorer (déjà exécutées) :")
        print("   – Collecte HAL (fetch_lab_publications)")
        print("   – Construction du DataFrame (df)")
        print("   – Enrichissement ORCID")
        print("ℹ️  Relancer à partir de la cellule OpenAlex (déduplication / couverture).")
else:
    print("ℹ️  Aucun fichier cache trouvé.")
    print("→ Exécutez les cellules de collecte HAL puis d'enrichissement ORCID.")


⚠️  Fichiers cache trouvés mais colonnes manquantes :
   hal_publications_uniques.csv : ['noms', 'prenoms', 'idhal']
→ Relancez les cellules de collecte HAL.


## Liste des laboratoires

In [3]:
LABS = [
    ("CMP",           {244423}),
    ("CRAN",          {185180, 1001}),
    ("CREATIS",       {139739}),
    ("CRIL",          {90448, 1628}),
    ("CRISTAL",       {410272, 389110, 388977, 390300, 183073, 111636, 24885, 2546, 186929}),
    ("DI ENS",        {25027}),
    ("CROSSING (IRL)",{1063106}),
    ("ETIS",          {1003474, 1061575, 1087906, 1003348}),
    ("FILOFOCS (UMI)",{1006288, 1006289}),
    ("GIPSA-Lab",     {1043333, 1042376, 24470}),
    ("GREYC",         {150}),
    ("G-SCOP",        {1043137, 1041927, 74240}),
    ("HEUDIASYC",     {389870}),
    ("I3S",           {13009, 552896, 1079434}),
    ("ICUBE",         {217648, 1073080}),
    ("IDRIS",         {1823}),
    ("IPAL (IRL)",    {542003, 220880, 138926}),
    ("IRIF",          {1005016, 444497}),
    ("IRISA",         {491183, 491231, 490899, 491188, 1092618, 491177, 1092619, 490900,
                       419364, 419370, 105128, 2494, 25255, 419365, 419367, 491230,
                       419363, 419366, 491232, 419362, 1099404, 545024, 1099406,
                       1099401, 1099402, 1099435, 525233, 1088566, 1088569, 495900,
                       489780, 1092631, 1092630, 1092628, 1092632, 1092626, 1092625, 1092629}),
    ("IRIT",          {34499, 1082335}),
    ("ISIR",          {541937, 96164}),
    ("JFLI",          {542009, 229050}),
    ("L2S",           {1051117, 1289}),
    ("LAAS",          {459}),
    ("LABRI",         {3102}),
    ("LAB-STICC",     {486345, 491660, 199324, 81533, 1089048}),
    ("LAMIH",         {1067790, 1303}),
    ("LAMSADE",       {989}),
    ("LIG",           {1043301, 1041964, 24471}),
    ("LIGM",          {1001627, 3210}),
    ("LIMOS",         {1063677, 490706, 857}),
    ("LIP",           {35418}),
    ("LIP6",          {541703, 233, 1095103}),
    ("LIPN",          {1000994, 994, 1086916, 1056718}),
    ("LIRIS",         {2003, 1086665}),
    ("LIRMM",         {181, 1071941}),
    ("LIS",           {527033, 199402, 199394, 862, 178374}),
    ("LISN",          {1061259, 1041968, 247329, 2544, 1050003, 81750}),
    ("LIX",           {2071, 1041697, 1071530, 1070048}),
    ("LMF",           {1065710}),
    ("LORIA",         {206040, 466633}),
    ("LS2N",          {1088564, 473973, 95421, 1693, 21439}),
    ("LMF-LSV",       {1065710, 1042689, 2571}),
    ("MDLS",          {210816}),
    ("RELAX",         {1040410, 528907}),
    ("ROOT (IRL)",    {389700}),
    ("STMS",          {541779, 1374}),
    ("TIMA",          {1043043, 1044023, 640}),
    ("TIMC IMAG",     {1043049, 1070489, 1042061, 707, 574002, 555959, 1056575}),
    ("VERIMAG",       {1043148, 1041816, 194}),
]

In [4]:
# LABS = [
#     ("CMP",           {244423}),
#     ("CRAN",          {185180, 1001})
# ]

## Fonction d'interrogation de l'API HAL

In [5]:
HAL_API_URL = "https://api.archives-ouvertes.fr/search/"
HAL_REF_URL = "https://api.archives-ouvertes.fr/ref/author/"
YEAR_MIN    = 2015
YEAR_MAX    = 2025
BATCH_SIZE  = 1000
SLEEP_SEC   = 0.3


def _hal_scalar(val):
    """Retourne le premier élément si HAL renvoie une liste, sinon la valeur brute."""
    if isinstance(val, list):
        return val[0] if val else None
    return val


def _pad(lst: list, length: int, fill="") -> list:
    """Complète une liste jusqu'à `length` avec `fill`."""
    return list(lst) + [fill] * max(0, length - len(lst))


def fetch_lab_publications(lab_name: str, struct_ids: set,
                           year_min: int = YEAR_MIN,
                           year_max: int = YEAR_MAX) -> list[dict]:
    """
    Récupère toutes les publications HAL d'un laboratoire.
    Champs : halId, noms, prénoms, idHal, DOI, année, type doc, langue,
             éditeur, revue, conférence.
    La colonne orcids est initialisée vide et remplie après par fetch_orcids().
    """
    struct_query = " OR ".join(str(sid) for sid in struct_ids)
    q  = f"structId_i:({struct_query})"
    fq = f"producedDateY_i:[{year_min} TO {year_max}]"

    records = []
    start   = 0

    while True:
        params = {
            "q":     q,
            "fq":    fq,
            "fl":    ("halId_s,title_s,"
                      "authLastName_s,authFirstName_s,authIdHal_s,doiId_s,"
                      "producedDateY_i,docType_s,language_s,"
                      "publisher_s,journalTitle_s,conferenceTitle_s"),
            "rows":  BATCH_SIZE,
            "start": start,
            "wt":    "json",
        }

        resp = requests.get(HAL_API_URL, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()

        docs      = data["response"]["docs"]
        num_found = data["response"]["numFound"]

        for doc in docs:
            title_field = doc.get("title_s")
            lang_field  = doc.get("language_s")

            last_names  = doc.get("authLastName_s")  or []
            first_names = doc.get("authFirstName_s") or []
            idhals      = doc.get("authIdHal_s")     or []

            n = max(len(last_names), len(first_names), len(idhals))
            last_names  = _pad(last_names,  n)
            first_names = _pad(first_names, n)
            idhals      = _pad(idhals,      n)

            full_names = [
                f"{fn} {ln}".strip() for fn, ln in zip(first_names, last_names)
            ]

            records.append({
                "laboratoire":   lab_name,
                "auteurs":       "; ".join(full_names),
                "noms":          "; ".join(last_names),
                "prenoms":       "; ".join(first_names),
                "idhal":         "; ".join(str(x) if x else "" for x in idhals),
                "orcids":        "",   # rempli après par fetch_orcids()
                "titre":         title_field[0] if title_field else None,
                "doi":           doc.get("doiId_s"),
                "annee":         doc.get("producedDateY_i"),
                "type_doc":      doc.get("docType_s"),
                "langue":        lang_field[0] if lang_field else None,
                "editeur":       _hal_scalar(doc.get("publisher_s")),
                "journal_revue": _hal_scalar(doc.get("journalTitle_s")),
                "conference":    _hal_scalar(doc.get("conferenceTitle_s")),
                "hal_id":        doc.get("halId_s"),
            })

        start += len(docs)
        if start >= num_found or not docs:
            break

        time.sleep(SLEEP_SEC)

    return records

## Collecte des données pour tous les laboratoires

In [ ]:
all_records = []
errors      = []

for lab_name, struct_ids in tqdm(LABS, desc="Laboratoires"):
    try:
        records = fetch_lab_publications(lab_name, struct_ids)
        all_records.extend(records)
        print(f"  {lab_name:20s} → {len(records):>5} publications")
    except Exception as e:
        errors.append((lab_name, str(e)))
        print(f"  ⚠ Erreur pour {lab_name}: {e}")

print(f"\nTotal brut : {len(all_records)} enregistrements")
if errors:
    print(f"Laboratoires en erreur : {[e[0] for e in errors]}")

Laboratoires:   0%|          | 0/50 [00:00<?, ?it/s]

  CMP                  →    14 publications
  CRAN                 →  4053 publications
  CREATIS              →  3413 publications
  CRIL                 →   814 publications
  CRISTAL              →  7209 publications
  DI ENS               →  2543 publications
  CROSSING (IRL)       →   158 publications
  ETIS                 →  1603 publications
  FILOFOCS (UMI)       →     1 publications
  GIPSA-Lab            →  5268 publications
  GREYC                →  2130 publications
  G-SCOP               →  2085 publications
  HEUDIASYC            →  1854 publications
  I3S                  →  3719 publications
  ICUBE                →  8268 publications
  IDRIS                →    36 publications
  IPAL (IRL)           →   237 publications
  IRIF                 →  1606 publications
  IRISA                → 11745 publications
  IRIT                 →  7716 publications
  ISIR                 →  2255 publications
  JFLI                 →   179 publications
  L2S                  →  5619 p

## Construction du dataframe global

In [9]:
DF_COLS = ["laboratoire", "auteurs", "noms", "prenoms", "idhal", "orcids",
           "titre", "doi", "annee", "type_doc", "langue",
           "editeur", "journal_revue", "conference", "hal_id"]

df = pd.DataFrame(all_records, columns=DF_COLS)
df["annee"] = pd.to_numeric(df["annee"], errors="coerce").astype("Int64")

print(f"Shape      : {df.shape}")
print(f"Années     : {sorted(df['annee'].dropna().unique().tolist())}")
print(f"Labos      : {df['laboratoire'].nunique()}")
print(f"Types doc  : {sorted(df['type_doc'].dropna().unique().tolist())}")
print(f"Langues    : {sorted(df['langue'].dropna().unique().tolist())}")
n_with_idhal = df["idhal"].apply(lambda x: any(i.strip() for i in (x or "").split(";"))).sum()
print(f"Pubs avec au moins 1 idHal : {n_with_idhal}")
df.head(10)

Shape      : (161590, 15)
Années     : [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Labos      : 50
Types doc  : ['ART', 'BLOG', 'COMM', 'COUV', 'HDR', 'IMG', 'ISSUE', 'LECTURE', 'MAP', 'MEM', 'NOTICE', 'OTHER', 'OUV', 'PATENT', 'POSTER', 'PRESCONF', 'PROCEEDINGS', 'REPORT', 'SOFTWARE', 'SON', 'SYNTHESE', 'THESE', 'TRAD', 'UNDEFINED', 'VIDEO']
Langues    : ['br', 'de', 'en', 'es', 'fr', 'hu', 'id', 'it', 'ja', 'ko', 'mk', 'pl', 'pt', 'ru', 'sq', 'tr', 'uk', 'und', 'zh']
Pubs avec au moins 1 idHal : 136687


,laboratoire,auteurs,noms,prenoms,idhal,orcids,titre,doi,annee,type_doc,langue,editeur,journal_revue,conference,hal_id
0,CMP,Aabir Chouichi; Jakey Blue; Claude Yugma; Fran...,Chouichi; Blue; Yugma; Pasqualini,Aabir; Jakey; Claude; François,jakey-blue; ; ;,,THE DETECTION AND THE CONTROL OF MACHINE/CHAMB...,NaN,2018,COMM,en,NaN,NaN,Winter Simulation Conference (WSC) 2018,hal-02067544
1,CMP,Matthieu Raoux; Éléonore Bertin; Antoine Pirog...,Raoux; Bertin; Pirog; Perrier; Jaffredo; Villa...,Matthieu; Éléonore; Antoine; Romain; Manon; Ar...,matthieu-raoux; antoine-pirog; romain-perrier;...,,Décodage des algorithmes endogènes des îlots e...,10.1016/S1262-3636(17)30122-2,2017,COMM,fr,NaN,NaN,Société Francophone du Diabète (SFD),hal-02520987
2,CMP,Asma Rakiz; Nabil Absi; Pierre Féniès,Rakiz; Absi; Féniès,Asma; Nabil; Pierre,asma-rakiz; nabil-absi;,,Comparing approaches for a multi-level plannin...,10.1016/j.ijpe.2023.108999,2023,ART,en,NaN,International Journal of Production Economics,NaN,hal-04461328
3,CMP,El Agharben; M. Bileci; A. Roussy; Marc Bocquet,Agharben; Bileci; Roussy; Bocquet,El; M.; A.; Marc,agnes-roussy; bocquetm; ;,,Flash gate optimized process and integration f...,10.1109/ISSM.2016.7934533,2016,COMM,en,IEEE,NaN,2016 International Symposium on Semiconductor ...,hal-01737950
4,CMP,Omar Kassem; Mohamed Saadaoui; Mathilde Rieu; ...,Kassem; Saadaoui; Rieu; Viricelle,Omar; Mohamed; Mathilde; Jean-Paul,mathilde-rieu; jpviricelle; ;,,Synthesis and Inkjet Printing of SnO<sub>2</su...,10.3390/proceedings1040622,2017,COMM,en,MDPI,NaN,EUROSENSORS 2017,hal-01617448
5,CMP,Maria Papaiordanidou; Seiichi Takamatsu; Shaha...,Papaiordanidou; Takamatsu; Rezaei-Mazinani; Lo...,Maria; Seiichi; Shahab; Thomas; Alain; Esma,maria-papaiordanidou; ; ; ; ;,,Cutaneous Recording and Stimulation of Muscles...,10.1002/adhm.201600299,2016,ART,en,NaN,Advanced Healthcare Materials,NaN,hal-01387198
6,CMP,El Agharben; A. Roussy; Marc Bocquet; M. Bilec...,Agharben; Roussy; Bocquet; Bileci; Bégouin; Ma...,El; A.; Marc; M.; S; A.,agnes-roussy; bocquetm; ; ; ;,,Critical sensitivity of flash gate dimension s...,10.1109/ASMC.2015.7164428,2015,COMM,en,IEEE,NaN,2015 26th Annual SEMI Advanced Semiconductor M...,hal-01737953
7,CMP,Laurent-Stéphane Didier; Nadia Mrabet; Léa Gla...,Didier; Mrabet; Glandus; Robert,Laurent-Stéphane; Nadia; Léa; Jean-Marc,nadia-el-mrabet; jean-marc-robert; ;,,Truncated multiplication and batch software SI...,10.62056/a3txl86bm,2024,ART,en,NaN,IACR Communications in Cryptology,NaN,hal-04742143
8,CMP,Eloise Bihar; Timothee Roberts; Mohamed Saadao...,Bihar; Roberts; Saadaoui; Herve; de Graaf; Mal...,Eloise; Timothee; Mohamed; Thierry; Jozina B.;...,jozina-de-graaf; ; ; ; ;,,Inkjet-Printed PEDOT:PSS Electrodes on Paper f...,10.1002/adhm.201601167,2017,ART,en,NaN,Advanced Healthcare Materials,NaN,hal-01691176
9,CMP,Dimitrios Koutsouras; Romain Perrier; Ariana V...,Koutsouras; Perrier; Villarroel Marquez; Pirog...,Dimitrios; Romain; Ariana; Antoine; Eileen; Er...,antoine-pirog; eric-cloutet; matthieu-raoux; j...,,Simultaneous monitoring of single cell and of ...,10.1016/j.msec.2017.07.028,2017,ART,en,NaN,Materials Science and Engineering: C,NaN,hal-01578781


In [ ]:
# ── Sauvegarde intermédiaire après collecte HAL (avant enrichissement ORCID) ──
_HAL_SAVE_COLS = ["laboratoire", "auteurs", "noms", "prenoms", "idhal", "orcids",
                  "titre", "doi", "annee", "type_doc", "langue",
                  "editeur", "journal_revue", "conference", "hal_id"]
df[_HAL_SAVE_COLS].to_csv("hal_publications_par_labo.csv", index=False, encoding="utf-8-sig")
print(f"✅ hal_publications_par_labo.csv  ({len(df):,} lignes — orcids non encore enrichis)")

## Enrichissement ORCID via le référentiel auteur HAL

Deux passes :
1. **Par idHal** — lookup direct et fiable dans `https://api.archives-ouvertes.fr/ref/author/`
2. **Par nom/prénom** — pour les auteurs sans idHal (moins fiable : on ne retient le résultat que si la correspondance est unique)

In [ ]:
REF_BATCH = 50    # idHals par requête ref/author
REF_SLEEP = 0.2   # pause entre requêtes


def _str(val) -> str:
    """Garantit une chaîne, même si HAL a retourné une liste."""
    if isinstance(val, list):
        return val[0] if val else ""
    return str(val) if val else ""


def _bare_orcid(orcid: str) -> str:
    """Normalise un ORCID au format nu : '0000-0001-xxxx-xxxx' (sans URI)."""
    orcid = orcid.strip()
    for prefix in ("https://orcid.org/", "http://orcid.org/", "orcid.org/"):
        if orcid.startswith(prefix):
            orcid = orcid[len(prefix):]
    return orcid


# ── Collecte de tous les idHal uniques présents dans df ──────────────────────
all_idhals = set()
for row in df["idhal"].dropna():
    for idh in _str(row).split(";"):
        idh = idh.strip()
        if idh:
            all_idhals.add(idh)

all_idhals_list = sorted(all_idhals)
print(f"idHal uniques trouvés dans df : {len(all_idhals_list)}")

# ── Passe 1 : ORCID par idHal via ref/author ─────────────────────────────────
idhal_to_orcid = {}   # {idhal -> orcid_nu}

for i in tqdm(range(0, len(all_idhals_list), REF_BATCH),
              total=-(-len(all_idhals_list) // REF_BATCH),
              desc="Ref/author idHal", unit="lot"):
    batch = all_idhals_list[i : i + REF_BATCH]
    q = "idHal_s:(" + " OR ".join(f'"{idh}"' for idh in batch) + ")"
    params = {"q": q, "fl": "idHal_s,orcidId_s", "rows": REF_BATCH, "wt": "json"}
    try:
        resp = requests.get(HAL_REF_URL, params=params, timeout=30)
        resp.raise_for_status()
        for doc in resp.json().get("response", {}).get("docs", []):
            idh   = _str(doc.get("idHal_s")).strip()
            orcid = _bare_orcid(_str(doc.get("orcidId_s")))   # ← format nu
            if idh and orcid:
                idhal_to_orcid[idh] = orcid
    except Exception as e:
        print(f"  ⚠ Erreur lot {i} : {e}")
    time.sleep(REF_SLEEP)

print(f"ORCIDs trouvés via idHal : {len(idhal_to_orcid)} / {len(all_idhals_list)}")

# ── Mise à jour de la colonne orcids dans df (format nu) ─────────────────────
def build_orcids_row(row) -> str:
    idhals = [_str(x) for x in _str(row["idhal"]).split(";")]
    noms   = [_str(x) for x in _str(row["noms"]).split(";")]
    n      = max(len(idhals), len(noms))
    idhals = _pad(idhals, n)
    orcids = [idhal_to_orcid.get(idh.strip(), "") for idh in idhals]
    return "; ".join(orcids)

df["orcids"] = df.apply(build_orcids_row, axis=1)

n_pubs_with_orcid = (df["orcids"].str.replace(";", "").str.strip() != "").sum()
print(f"Publications avec au moins 1 ORCID : {n_pubs_with_orcid}")

In [ ]:
# ── Sauvegarde après enrichissement ORCID ─────────────────────────────────────
_HAL_SAVE_COLS = ["laboratoire", "auteurs", "noms", "prenoms", "idhal", "orcids",
                  "titre", "doi", "annee", "type_doc", "langue",
                  "editeur", "journal_revue", "conference", "hal_id"]
df[_HAL_SAVE_COLS].to_csv("hal_publications_par_labo.csv", index=False, encoding="utf-8-sig")
n_orcid = df["orcids"].dropna().astype(str).str.strip().ne("").sum()
print(f"✅ hal_publications_par_labo.csv  ({len(df):,} lignes, {n_orcid:,} avec ORCID)")

In [16]:
df["orcids"]

0                                                    ; ; ; 
1         ; ; ; https://orcid.org/0000-0002-5616-2979; ;...
2         https://orcid.org/0000-0002-6371-8664; https:/...
3               ; https://orcid.org/0000-0003-3777-5793; ; 
4               ; https://orcid.org/0000-0002-7293-6591; ; 
                                ...                        
161585    https://orcid.org/0000-0001-5765-0758; https:/...
161586                                                     
161587            https://orcid.org/0000-0001-9267-7593; ; 
161588            https://orcid.org/0000-0001-9267-7593; ; 
161589            https://orcid.org/0000-0001-9267-7593; ; 
Name: orcids, Length: 161590, dtype: str

In [18]:
df.head()

,laboratoire,auteurs,noms,prenoms,idhal,orcids,titre,doi,annee,type_doc,langue,editeur,journal_revue,conference,hal_id
0,CMP,Aabir Chouichi; Jakey Blue; Claude Yugma; Fran...,Chouichi; Blue; Yugma; Pasqualini,Aabir; Jakey; Claude; François,jakey-blue; ; ;,; ; ;,THE DETECTION AND THE CONTROL OF MACHINE/CHAMB...,NaN,2018,COMM,en,NaN,NaN,Winter Simulation Conference (WSC) 2018,hal-02067544
1,CMP,Matthieu Raoux; Éléonore Bertin; Antoine Pirog...,Raoux; Bertin; Pirog; Perrier; Jaffredo; Villa...,Matthieu; Éléonore; Antoine; Romain; Manon; Ar...,matthieu-raoux; antoine-pirog; romain-perrier;...,; ; ; https://orcid.org/0000-0002-5616-2979; ;...,Décodage des algorithmes endogènes des îlots e...,10.1016/S1262-3636(17)30122-2,2017,COMM,fr,NaN,NaN,Société Francophone du Diabète (SFD),hal-02520987
2,CMP,Asma Rakiz; Nabil Absi; Pierre Féniès,Rakiz; Absi; Féniès,Asma; Nabil; Pierre,asma-rakiz; nabil-absi;,https://orcid.org/0000-0002-6371-8664; https:/...,Comparing approaches for a multi-level plannin...,10.1016/j.ijpe.2023.108999,2023,ART,en,NaN,International Journal of Production Economics,NaN,hal-04461328
3,CMP,El Agharben; M. Bileci; A. Roussy; Marc Bocquet,Agharben; Bileci; Roussy; Bocquet,El; M.; A.; Marc,agnes-roussy; bocquetm; ;,; https://orcid.org/0000-0003-3777-5793; ;,Flash gate optimized process and integration f...,10.1109/ISSM.2016.7934533,2016,COMM,en,IEEE,NaN,2016 International Symposium on Semiconductor ...,hal-01737950
4,CMP,Omar Kassem; Mohamed Saadaoui; Mathilde Rieu; ...,Kassem; Saadaoui; Rieu; Viricelle,Omar; Mohamed; Mathilde; Jean-Paul,mathilde-rieu; jpviricelle; ;,; https://orcid.org/0000-0002-7293-6591; ;,Synthesis and Inkjet Printing of SnO<sub>2</su...,10.3390/proceedings1040622,2017,COMM,en,MDPI,NaN,EUROSENSORS 2017,hal-01617448


## Dédoublonnage + restauration des résultats OpenAlex

Les résultats OpenAlex sont **restaurés depuis le fichier `hal_openalex_uniques.csv`** (déjà calculé) sans relancer les requêtes API.

In [19]:
# ── Publications uniques ─────────────────────────────────────────────────────
df_unique = df.drop_duplicates(subset=["hal_id"]).reset_index(drop=True)
print(f"Publications uniques : {len(df_unique)}")

# ── Restauration des résultats OpenAlex depuis le CSV existant ───────────────
OA_CACHE  = "hal_openalex_uniques.csv"
OA_RESULT_COLS = ["hal_id", "in_openalex", "match_method",
                  "openalex_id", "openalex_title", "title_sim"]

try:
    oa_cache = pd.read_csv(
        OA_CACHE,
        usecols=lambda c: c in OA_RESULT_COLS,
        encoding="utf-8-sig",
        dtype={"in_openalex": "boolean"},
    )
    df_unique = df_unique.merge(oa_cache, on="hal_id", how="left")
    df_unique["in_openalex"] = df_unique["in_openalex"].fillna(False)
    n = df_unique["in_openalex"].sum()
    print(f"Résultats OpenAlex restaurés depuis '{OA_CACHE}' : {n} publications trouvées")
except FileNotFoundError:
    print(f"⚠  '{OA_CACHE}' introuvable — exécuter la section OpenAlex pour le générer.")
    df_unique["in_openalex"]    = False
    df_unique["match_method"]   = pd.NA
    df_unique["openalex_id"]    = pd.NA
    df_unique["openalex_title"] = pd.NA
    df_unique["title_sim"]      = pd.NA

# ── Propagation vers df (par labo) ───────────────────────────────────────────
OA_JOIN_COLS = ["hal_id", "in_openalex", "match_method", "openalex_id"]
df = df.drop(columns=[c for c in OA_JOIN_COLS[1:] if c in df.columns], errors="ignore")
df = df.merge(df_unique[OA_JOIN_COLS], on="hal_id", how="left")

print(f"df shape après enrichissement : {df.shape}")

Publications uniques : 120087
Résultats OpenAlex restaurés depuis 'hal_openalex_uniques.csv' : 51306 publications trouvées
df shape après enrichissement : (161590, 18)


In [ ]:
# ── Sauvegarde hal_publications_uniques (données HAL seules, dédoublonnées) ───
_HAL_SAVE_COLS = ["laboratoire", "auteurs", "noms", "prenoms", "idhal", "orcids",
                  "titre", "doi", "annee", "type_doc", "langue",
                  "editeur", "journal_revue", "conference", "hal_id"]
df_unique[_HAL_SAVE_COLS].to_csv("hal_publications_uniques.csv", index=False, encoding="utf-8-sig")
print(f"✅ hal_publications_uniques.csv   ({len(df_unique):,} lignes)")

---
# Croisement avec OpenAlex

Stratégie en deux passes sur `df_unique` :
1. **Passe DOI** : requête par lot (50 DOI à la fois) via `filter=doi:…`
2. **Passe titre** : pour les publications sans DOI (ou non trouvées par DOI), recherche sur `display_name.search` avec vérification de similarité (seuil 0.85)

> **Note** : la recherche par titre est heuristique — des faux positifs sont possibles sur des titres courts ou très génériques.
>
> Si les résultats OpenAlex ont déjà été calculés et exportés, **ne pas ré-exécuter les cellules de cette section** — les résultats sont restaurés automatiquement depuis le CSV.

In [ ]:
import difflib
import unicodedata
import re
import math

# ── Paramètres OpenAlex ──────────────────────────────────────────────────────
OPENALEX_URL    = "https://api.openalex.org/works"
OPENALEX_MAILTO = "votre.email@example.com"   # remplacer pour le « polite pool »
OA_BATCH_SIZE   = 50     # max DOI par requête (recommandé par OpenAlex)
OA_SLEEP_SEC    = 0.1    # pause entre requêtes
TITLE_SIM_MIN   = 0.85   # seuil de similarité pour la correspondance par titre


def _norm_title(title: str) -> str:
    """Normalise un titre : minuscules, sans accents ni ponctuations."""
    if not title:
        return ""
    t = unicodedata.normalize("NFD", title.lower())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def _norm_doi(doi: str) -> str:
    """Renvoie le DOI sans le préfixe URI, en minuscules."""
    if not doi:
        return ""
    doi = doi.strip().lower()
    for prefix in ("https://doi.org/", "http://doi.org/", "doi:"):
        if doi.startswith(prefix):
            doi = doi[len(prefix):]
    return doi


def _bare_orcid(orcid: str) -> str:
    """Normalise un ORCID au format nu '0000-0001-xxxx-xxxx' (sans URI)."""
    orcid = str(orcid).strip()
    for prefix in ("https://orcid.org/", "http://orcid.org/", "orcid.org/"):
        if orcid.startswith(prefix):
            orcid = orcid[len(prefix):]
    return orcid


def _oa_params(**extra) -> dict:
    return {"mailto": OPENALEX_MAILTO, "select": "id,doi,display_name", **extra}


def lookup_dois(doi_list: list[str]) -> dict[str, dict]:
    """Recherche par lots de DOIs avec barre de progression. Renvoie {doi_normalisé -> {...}}."""
    results = {}
    n_batches = math.ceil(len(doi_list) / OA_BATCH_SIZE)
    for i in tqdm(range(0, len(doi_list), OA_BATCH_SIZE),
                  total=n_batches, desc="Lots DOI OpenAlex", unit="lot"):
        batch = doi_list[i : i + OA_BATCH_SIZE]
        filter_val = "|".join(batch)
        params = _oa_params(filter=f"doi:{filter_val}", per_page=OA_BATCH_SIZE)
        try:
            resp = requests.get(OPENALEX_URL, params=params, timeout=30)
            resp.raise_for_status()
            for work in resp.json().get("results", []):
                raw_doi = work.get("doi") or ""
                nd = _norm_doi(raw_doi)
                if nd:
                    results[nd] = {
                        "openalex_id":    work.get("id"),
                        "openalex_title": work.get("display_name"),
                    }
        except Exception as e:
            print(f"  ⚠ Erreur batch DOI [{i}:{i+OA_BATCH_SIZE}] : {e}")
        time.sleep(OA_SLEEP_SEC)
    return results


def lookup_title(title: str) -> dict | None:
    """Recherche par titre avec vérification de similarité."""
    if not title or len(title.strip()) < 10:
        return None
    search_title = title[:200]
    params = _oa_params(**{"filter": f'display_name.search:"{search_title}"', "per_page": 1})
    try:
        resp = requests.get(OPENALEX_URL, params=params, timeout=30)
        resp.raise_for_status()
        results = resp.json().get("results", [])
        if not results:
            return None
        work = results[0]
        oa_title = work.get("display_name") or ""
        sim = difflib.SequenceMatcher(
            None, _norm_title(title), _norm_title(oa_title)
        ).ratio()
        if sim >= TITLE_SIM_MIN:
            return {
                "openalex_id":    work.get("id"),
                "openalex_title": oa_title,
                "title_sim":      round(sim, 3),
            }
    except Exception as e:
        print(f"  ⚠ Erreur titre : {e}")
    return None

In [22]:
# ── Initialisation des colonnes résultats ────────────────────────────────────
df_unique = df_unique.copy()
df_unique["in_openalex"]    = False
df_unique["match_method"]   = pd.NA
df_unique["openalex_id"]    = pd.NA
df_unique["openalex_title"] = pd.NA
df_unique["title_sim"]      = pd.NA

# ── Passe 1 : DOI ────────────────────────────────────────────────────────────
has_doi = df_unique["doi"].notna() & (df_unique["doi"].str.strip() != "")
dois_norm = df_unique.loc[has_doi, "doi"].apply(_norm_doi).tolist()
doi_index = df_unique.loc[has_doi].index

print(f"Publications avec DOI : {has_doi.sum()} / {len(df_unique)}")
print("Recherche par DOI dans OpenAlex…")

doi_results = lookup_dois(dois_norm)
print(f"  → {len(doi_results)} DOI trouvés dans OpenAlex")

for idx, doi_raw in zip(doi_index, dois_norm):
    if doi_raw in doi_results:
        hit = doi_results[doi_raw]
        df_unique.at[idx, "in_openalex"]    = True
        df_unique.at[idx, "match_method"]   = "doi"
        df_unique.at[idx, "openalex_id"]    = hit["openalex_id"]
        df_unique.at[idx, "openalex_title"] = hit["openalex_title"]

print(f"Publications trouvées via DOI : {(df_unique['match_method'] == 'doi').sum()}")

Publications avec DOI : 63335 / 120087
Recherche par DOI dans OpenAlex…


Lots DOI OpenAlex:   0%|          | 0/1267 [00:00<?, ?lot/s]

  ⚠ Erreur batch DOI [50150:50200] : 400 Client Error: Bad Request for url: https://api.openalex.org/works?mailto=votre.email%40example.com&select=id%2Cdoi%2Cdisplay_name&filter=doi%3A10.1007%2F978-3-319-32360-2_12%7C10.1109%2Fisbi.2016.7493259%7C10.3722%2Fcadaps.2013.xxx-yyy%7C10.3233%2Ffaia240619%7C10.1109%2Ftase.2025.3540744%7C10.1101%2F2024.02.27.582302%7C10.4204%2Feptcs.439.21%7C10.1007%2F978-3-032-09544-2_12%7C10.4230%2Flipics.cp.2025.11%7C10.1111%2Fcogs.70095%7C10.1007%2Fs00170-015-7100-8%7C10.4000%2Fcorpus.2936%7C10.1007%2Fs10055-016-0302-z%7C10.1093%2Fcercor%2Fbhv123%7C10.1007%2Fs00453-014-9939-8%7C10.1159%2F000435825%7C10.18653%2Fv1%2Fw16-3621%7C10.1145%2F2933575.2935315%7C10.1007%2F978-3-319-46909-6_9%7C10.1016%2Fj.epsr.2016.11.021%7C10.1016%2Fj.nahs.2020.100950%7C10.7717%2Fpeerj-cs.466%7C10.1007%2Fs11517-015-1387-3%7C10.1103%2Fphysrevb.106.104309%7C10.48556%2Fsif.1024.21.83%7C10.1088%2F1475-7516%2F2024%2F09%2F042%7C10.1016%2Fj.procs.2024.09.362%7C10.1145%2F3559106%7C10.1145

In [ ]:
# ── Passe 2 : titre ──────────────────────────────────────────────────────────
to_search_by_title = df_unique[
    (~df_unique["in_openalex"]) & (df_unique["titre"].notna())
].index

print(f"Publications à rechercher par titre : {len(to_search_by_title)}")
print("Recherche par titre dans OpenAlex (peut être long)…")

for idx in tqdm(to_search_by_title, desc="Recherche titre"):
    titre = df_unique.at[idx, "titre"]
    hit = lookup_title(titre)
    if hit:
        df_unique.at[idx, "in_openalex"]    = True
        df_unique.at[idx, "match_method"]   = "titre"
        df_unique.at[idx, "openalex_id"]    = hit["openalex_id"]
        df_unique.at[idx, "openalex_title"] = hit["openalex_title"]
        df_unique.at[idx, "title_sim"]      = hit["title_sim"]
    time.sleep(OA_SLEEP_SEC)

print(f"Publications trouvées via titre : {(df_unique['match_method'] == 'titre').sum()}")
print(f"Publications non trouvées : {(~df_unique['in_openalex']).sum()}")

In [ ]:
# ── Sauvegarde des résultats OpenAlex ─────────────────────────────────────────
df_unique.to_csv("hal_openalex_uniques.csv",  index=False, encoding="utf-8-sig")
df.to_csv(       "hal_openalex_par_labo.csv", index=False, encoding="utf-8-sig")
n_found = int(df_unique["in_openalex"].sum())
print(f"✅ hal_openalex_uniques.csv   ({len(df_unique):,} lignes, {n_found:,} dans OpenAlex)")
print(f"✅ hal_openalex_par_labo.csv  ({len(df):,} lignes)")

## Taux de couverture global (df_unique)

In [23]:
total   = len(df_unique)
found   = df_unique["in_openalex"].sum()
by_doi  = (df_unique["match_method"] == "doi").sum()
by_titl = (df_unique["match_method"] == "titre").sum()

print("=" * 50)
print(f"Publications HAL uniques           : {total:>7}")
print(f"Présentes dans OpenAlex            : {found:>7}  ({100*found/total:.1f}%)")
print(f"  dont trouvées par DOI            : {by_doi:>7}  ({100*by_doi/total:.1f}%)")
print(f"  dont trouvées par titre          : {by_titl:>7}  ({100*by_titl/total:.1f}%)")
print(f"Non trouvées dans OpenAlex         : {total-found:>7}  ({100*(total-found)/total:.1f}%)")
print("=" * 50)

df_unique.groupby("match_method", dropna=False).size().rename("count").to_frame()

Publications HAL uniques           :  120087
Présentes dans OpenAlex            :   62357  (51.9%)
  dont trouvées par DOI            :   62357  (51.9%)
  dont trouvées par titre          :       0  (0.0%)
Non trouvées dans OpenAlex         :   57730  (48.1%)


,count
match_method,
doi,62357
NaN,57730


## Taux de couverture par laboratoire

In [24]:
# On repart de df (labo × publication) et on joint df_unique (résultats OA)
# pour être sûr d'avoir les bonnes valeurs de match_method et in_openalex.
df_labo = (
    df[["laboratoire", "hal_id", "doi"]]
    .merge(
        df_unique[["hal_id", "in_openalex", "match_method"]],
        on="hal_id", how="left"
    )
    .assign(_has_doi=lambda d: d["doi"].notna() & (d["doi"].str.strip() != ""))
)

stats_labo = (
    df_labo
    .groupby("laboratoire")
    .agg(
        n_hal      =("hal_id",       "count"),
        n_openalex =("in_openalex",  "sum"),
        n_doi      =("match_method", lambda x: (x == "doi").sum()),
        n_titre    =("match_method", lambda x: (x == "titre").sum()),
        n_avec_doi =("_has_doi",     "sum"),
        n_sans_doi =("_has_doi",     lambda x: (~x).sum()),
    )
    .assign(
        taux_global=lambda d: (d["n_openalex"] / d["n_hal"]      * 100).round(1),
        taux_doi   =lambda d: (d["n_doi"]      / d["n_avec_doi"] * 100).round(1),
        taux_titre =lambda d: (d["n_titre"]    / d["n_sans_doi"] * 100).round(1),
    )
    .sort_values("taux_global", ascending=False)
)

# Vérification : le total n_titre doit être >= (df_unique['match_method']=='titre').sum()
print(f"Total n_titre dans stats_labo : {stats_labo['n_titre'].sum()}")
print(f"Référence df_unique           : {(df_unique['match_method'] == 'titre').sum()}")
stats_labo

Total n_titre dans stats_labo : 0
Référence df_unique           : 0


,n_hal,n_openalex,n_doi,n_titre,n_avec_doi,n_sans_doi,taux_global,taux_doi,taux_titre
laboratoire,,,,,,,,,
CMP,14,11,11,0,11,3,78.6,100.0,0.0
RELAX,31,24,24,0,26,5,77.4,92.3,0.0
MDLS,370,282,282,0,282,88,76.2,100.0,0.0
L2S,5619,4116,4116,0,4133,1486,73.3,99.6,0.0
TIMC IMAG,3442,2518,2518,0,2526,916,73.2,99.7,0.0
IDRIS,36,24,24,0,24,12,66.7,100.0,0.0
CROSSING (IRL),158,104,104,0,104,54,65.8,100.0,0.0
LIGM,2293,1492,1492,0,1514,779,65.1,98.5,0.0
LIRMM,4111,2616,2616,0,2678,1433,63.6,97.7,0.0


In [ ]:
stats_labo.to_csv("couverture_par_labo.csv", encoding="utf-8-sig")
print("✅ couverture_par_labo.csv")

---
# Analyses de couverture par dimension

Les analyses suivantes utilisent `df_unique` (publications dédoublonnées) pour éviter de compter plusieurs fois une même publication affectée à plusieurs laboratoires.

## Couverture par année de publication

In [25]:
def coverage_table(df_: pd.DataFrame, col: str) -> pd.DataFrame:
    """Calcule le taux de couverture OpenAlex pour chaque valeur d'une colonne.
    - taux_global : % de publications HAL trouvées dans OpenAlex (/ n_hal)
    - taux_doi    : parmi les pubs HAL avec DOI, % retrouvées dans OpenAlex (/ n_avec_doi)
    - taux_titre  : parmi les pubs HAL sans DOI, % retrouvées dans OpenAlex (/ n_sans_doi)
    """
    has_doi = df_["doi"].notna() & (df_["doi"].str.strip() != "")
    return (
        df_.assign(_has_doi=has_doi)
        .groupby(col, dropna=False)
        .agg(
            n_hal      =("hal_id",       "count"),
            n_openalex =("in_openalex",  "sum"),
            n_doi      =("match_method", lambda x: (x == "doi").sum()),
            n_titre    =("match_method", lambda x: (x == "titre").sum()),
            n_avec_doi =("_has_doi",     "sum"),
            n_sans_doi =("_has_doi",     lambda x: (~x).sum()),
        )
        .assign(
            taux_global=lambda d: (d["n_openalex"] / d["n_hal"]      * 100).round(1),
            taux_doi   =lambda d: (d["n_doi"]      / d["n_avec_doi"] * 100).round(1),
            taux_titre =lambda d: (d["n_titre"]    / d["n_sans_doi"] * 100).round(1),
        )
        .sort_index()
    )


stats_annee = coverage_table(df_unique, "annee")
print("Couverture OpenAlex par année de publication")
stats_annee

Couverture OpenAlex par année de publication


,n_hal,n_openalex,n_doi,n_titre,n_avec_doi,n_sans_doi,taux_global,taux_doi,taux_titre
annee,,,,,,,,,
2015,11683,5347,5347,0,5402,6281,45.8,99.0,0.0
2016,11924,5624,5624,0,5705,6219,47.2,98.6,0.0
2017,11763,5616,5616,0,5695,6068,47.7,98.6,0.0
2018,11787,5939,5939,0,6015,5772,50.4,98.7,0.0
2019,11582,6283,6283,0,6354,5228,54.2,98.9,0.0
2020,10085,5982,5982,0,6060,4025,59.3,98.7,0.0
2021,10839,6395,6395,0,6480,4359,59.0,98.7,0.0
2022,10874,5935,5935,0,6026,4848,54.6,98.5,0.0
2023,10656,5819,5819,0,5896,4760,54.6,98.7,0.0


In [ ]:
stats_annee.to_csv("couverture_par_annee.csv", encoding="utf-8-sig")
print("✅ couverture_par_annee.csv")

## Couverture par type de document (docType_s)

In [26]:
stats_type = (
    coverage_table(df_unique, "type_doc")
    .sort_values("taux_global", ascending=False)
)
print("Couverture OpenAlex par type de document")
stats_type

Couverture OpenAlex par type de document


,n_hal,n_openalex,n_doi,n_titre,n_avec_doi,n_sans_doi,taux_global,taux_doi,taux_titre
type_doc,,,,,,,,,
ART,38222,34695,34695,0,35076,3146,90.8,98.9,0.0
COUV,3471,2031,2031,0,2050,1421,58.5,99.1,0.0
ISSUE,330,167,167,0,172,158,50.6,97.1,0.0
PROCEEDINGS,1049,531,531,0,539,510,50.6,98.5,0.0
NOTICE,31,15,15,0,15,16,48.4,100.0,0.0
COMM,55229,23475,23475,0,23949,31280,42.5,98.0,0.0
OUV,1109,383,383,0,385,724,34.5,99.5,0.0
TRAD,29,7,7,0,7,22,24.1,100.0,0.0
UNDEFINED,4240,581,581,0,616,3624,13.7,94.3,0.0


In [ ]:
stats_type.to_csv("couverture_par_type_doc.csv", encoding="utf-8-sig")
print("✅ couverture_par_type_doc.csv")

## Couverture par langue de publication (language_s)

In [27]:
stats_langue = (
    coverage_table(df_unique, "langue")
    .sort_values("taux_global", ascending=False)
)
print("Couverture OpenAlex par langue de publication")
stats_langue

Couverture OpenAlex par langue de publication


,n_hal,n_openalex,n_doi,n_titre,n_avec_doi,n_sans_doi,taux_global,taux_doi,taux_titre
langue,,,,,,,,,
br,1,1,1,0,1,0,100.0,100.0,NaN
tr,1,1,1,0,1,0,100.0,100.0,NaN
en,104012,60983,60983,0,61871,42141,58.6,98.6,0.0
hu,2,1,1,0,1,1,50.0,100.0,0.0
es,63,19,19,0,19,44,30.2,100.0,0.0
de,11,3,3,0,3,8,27.3,100.0,0.0
pt,73,17,17,0,19,54,23.3,89.5,0.0
uk,5,1,1,0,1,4,20.0,100.0,0.0
ru,25,5,5,0,5,20,20.0,100.0,0.0


In [ ]:
stats_langue.to_csv("couverture_par_langue.csv", encoding="utf-8-sig")
print("✅ couverture_par_langue.csv")

---
## Récapitulatif des fichiers exportés

Les fichiers sont sauvegardés **au fur et à mesure** de l'analyse. La cellule ci-dessous vérifie leur présence sur le disque.

In [ ]:
import os

_FILES = [
    ("hal_publications_par_labo.csv",  "Publications HAL par labo (avec ORCID, non dédoublonné)"),
    ("hal_publications_uniques.csv",   "Publications HAL dédoublonnées (avec ORCID)"),
    ("hal_openalex_par_labo.csv",      "HAL + OpenAlex, par labo"),
    ("hal_openalex_uniques.csv",       "HAL + OpenAlex, dédoublonné"),
    ("couverture_par_labo.csv",        "Taux de couverture par laboratoire"),
    ("couverture_par_annee.csv",       "Taux de couverture par année"),
    ("couverture_par_type_doc.csv",    "Taux de couverture par type de document"),
    ("couverture_par_langue.csv",      "Taux de couverture par langue"),
]

print("Fichiers exportés :")
for fname, desc in _FILES:
    status = "✅" if os.path.exists(fname) else "❌ manquant"
    print(f"  {status}  {fname:45s}  {desc}")

---
# Analyse des auteurs d'un laboratoire pour une année donnée

Comparaison des auteurs obtenus via deux sources :
- **Solution A** : requête directe à OpenAlex sur l'institution
- **Solution B** : données HAL déjà collectées (`df`)

La cellule de comparaison identifie les auteurs présents dans HAL mais absents d'OpenAlex, et inversement.

In [29]:
# ── Paramètres ───────────────────────────────────────────────────────────────
LABO_CIBLE  = "LIS"
ANNEE_CIBLE = 2022

# ════════════════════════════════════════════════════════════════════════════
# SOLUTION A : auteurs via OpenAlex (requête directe sur l'institution)
# ════════════════════════════════════════════════════════════════════════════

# Étape A1 – chercher le laboratoire par son acronyme dans OpenAlex
resp = requests.get(
    "https://api.openalex.org/institutions",
    params={
        "search":   LABO_CIBLE,
        "filter":   "country_code:FR",
        "per_page": 10,
        "mailto":   OPENALEX_MAILTO,
    },
    timeout=30,
)
resp.raise_for_status()
institutions = resp.json().get("results", [])

print("Institutions trouvées — choisir l'ID à utiliser :")
for inst in institutions:
    print(f"  {inst['id']}  |  {inst['display_name']}  |  {inst.get('ror', '?')}")

# ── Mettre ici l'ID OpenAlex correspondant au laboratoire ────────────────────
# (ajuster si nécessaire après lecture des résultats ci-dessus)
OA_LIS_ID = institutions[0]["id"] if institutions else None
print(f"\nID retenu : {OA_LIS_ID}")

Institutions trouvées — choisir l'ID à utiliser :
  https://openalex.org/I4210114274  |  Laboratoire d’Informatique et Systèmes  |  https://ror.org/0257sgk90

ID retenu : https://openalex.org/I4210114274


In [30]:
# Étape A2 – récupérer tous les travaux de LIS en ANNEE_CIBLE et extraire les auteurs
# (cursor-based pagination pour éviter la limite de 10 000 résultats)
authors_A = {}   # {nom_normalisé: nom_original}
cursor = "*"
n_works = 0

while cursor:
    params = {
        "filter":   f"institutions.id:{OA_LIS_ID},publication_year:{ANNEE_CIBLE}",
        "select":   "id,authorships",
        "per_page": 200,
        "cursor":   cursor,
        "mailto":   OPENALEX_MAILTO,
    }
    resp = requests.get(OPENALEX_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    for work in data.get("results", []):
        n_works += 1
        for authorship in work.get("authorships", []):
            name = (authorship.get("author") or {}).get("display_name", "").strip()
            if name:
                authors_A[_norm_title(name)] = name   # _norm_title sert de clé normalisée

    cursor = data.get("meta", {}).get("next_cursor")
    time.sleep(OA_SLEEP_SEC)

authors_A_list = sorted(authors_A.values(), key=lambda n: n.split()[-1])  # tri par nom

print(f"Publications LIS {ANNEE_CIBLE} trouvées dans OpenAlex : {n_works}")
print(f"Auteurs uniques (Solution A) : {len(authors_A_list)}")
print()
for a in authors_A_list:
    print(f"  {a}")

Publications LIS 2022 trouvées dans OpenAlex : 372
Auteurs uniques (Solution A) : 1286

  Aubert, J. -J.
  Domi, A.
  Albert, A.
  Capone, A.
  Coleiro, A.
  Creusot, A.
  Enzenhöfer, A.
  Kouchner, A.
  Margiotta, A.
  Marinelli, A.
  Martínez-Mora, J. A.
  Mahdi Abbasi
  Jean Aboudarham
  Pierre Aboulker
  Luciano A. Abriata
  Rim Abrougui
  Luca Aceto
  Antonis Achilleos
  Olivier Adam
  Karine Adeline
  Hafiz Ahmed
  Wassim Ahmed-Belkacem
  Eunice Akani
  Mohammed Al-Kharaz
  Saïd Ouatik El Alaoui
  A. Albert
  Cécile H. Albert
  Emanuel Aldea
  Miquel Alfaras
  Mohammed Alhasheem
  Oihab Allal‐Chérif
  Sabri Allani
  Majedaldein Almahasneh
  André L. F. de Almeida
  J. A. Afonso de Almeida
  Lionel Alméras
  Alexandre Altes
  S. Alves
  Saïd Amari
  Souad Ameddah
  Rabah Ammour
  Bouchra Ananou
  Elli Anastasiadi
  Thomas Andrillon
  M. André
  M. Anghinolfi
  Jesús Angulo
  Cédric Anthierens
  Sandrine Anthoine
  Elie Antoine
  G. Anton
  Alba Martinez Anton
  Inés M. Antón
  Zac

In [31]:
# ════════════════════════════════════════════════════════════════════════════
# SOLUTION B : auteurs via les données HAL (df)
# ════════════════════════════════════════════════════════════════════════════

mask = (df["laboratoire"] == LABO_CIBLE) & (df["annee"] == ANNEE_CIBLE)
n_pubs_B = mask.sum()

authors_B = {}   # {nom_normalisé: nom_original}
for row in df.loc[mask, "auteurs"].dropna():
    for name in row.split("; "):
        name = name.strip()
        if name:
            authors_B[_norm_title(name)] = name

authors_B_list = sorted(authors_B.values(), key=lambda n: n.split()[-1])

print(f"Publications LIS {ANNEE_CIBLE} dans HAL : {n_pubs_B}")
print(f"Auteurs uniques (Solution B) : {len(authors_B_list)}")
print()
for a in authors_B_list:
    print(f"  {a}")

Publications LIS 2022 dans HAL : 280
Auteurs uniques (Solution B) : 1689

  Mahdi Abbasi
  R. Abbasi
  Nawal Abboub
  Pierre Aboulker
  P. Abreu
  Rim Abrougui
  T. Abu-Zayyad
  Antonis Achilleos
  M. Ackermann
  Olivier Adam
  J. Adams
  Karine Adeline
  Adaeze Adigwe
  M. Aglietta
  J.A. Aguilar
  M. Ahlers
  Mohamed Mahmoud Mohamed Ahmed
  Hafiz Ahmed
  Wassim Ahmed-Belkacem
  M. Ahrens
  Eunice Akani
  J.M. Alameddine
  A. Albert
  Cécile H. Albert
  J.M. Albury
  Emanuel Aldea
  Saud Aleissa
  Afra Alishahi
  C. Alispach
  Oihab Allal-Chérif
  I. Allekotte
  M. Allen
  R.M. de Almeida
  A. Almela
  Lionel Almeras
  J. Alvarez-Muñiz
  S. Alves
  Jr.A.A. Alves
  A.A. Alves
  Said Amari
  N.M. Amin
  Rabah Ammour
  G.A. Anastasi
  L. Anchordoqui
  K. Andeen
  T. Anderson
  B. Andrada
  S. Andringa
  M. André
  M. Anghinolfi
  R.C. dos Anjos
  Sandrine Anthoine
  Elie Antoine
  G. Anton
  Alba Martinez Anton
  Zacharie Aoulad
  Denis Apothéloz
  Y. Arai
  C. Aramo
  Emma de Araujo
  S

In [32]:
# ════════════════════════════════════════════════════════════════════════════
# COMPARAISON : dataframe de tous les auteurs avec présence HAL / OpenAlex
# ════════════════════════════════════════════════════════════════════════════

# Union de toutes les clés normalisées
all_keys = set(authors_A) | set(authors_B)

rows = []
for key in all_keys:
    rows.append({
        "auteur":      authors_B.get(key) or authors_A.get(key),  # nom préféré depuis HAL
        "dans_HAL":    key in authors_B,
        "dans_OpenAlex": key in authors_A,
    })

df_auteurs = (
    pd.DataFrame(rows)
    .sort_values(["dans_HAL", "dans_OpenAlex", "auteur"],
                 ascending=[False, False, True])
    .reset_index(drop=True)
)

print(f"Auteurs LIS {ANNEE_CIBLE} — récapitulatif")
print(f"  Dans HAL et OpenAlex  : {(df_auteurs['dans_HAL'] & df_auteurs['dans_OpenAlex']).sum()}")
print(f"  Dans HAL seulement    : {(df_auteurs['dans_HAL'] & ~df_auteurs['dans_OpenAlex']).sum()}")
print(f"  Dans OpenAlex seulement: {(~df_auteurs['dans_HAL'] & df_auteurs['dans_OpenAlex']).sum()}")
print(f"  Total                 : {len(df_auteurs)}")
df_auteurs

Auteurs LIS 2022 — récapitulatif
  Dans HAL et OpenAlex  : 546
  Dans HAL seulement    : 1143
  Dans OpenAlex seulement: 740
  Total                 : 2429


,auteur,dans_HAL,dans_OpenAlex
0,A. Albert,True,True
1,A. Capone,True,True
2,A. Coleiro,True,True
3,A. Creusot,True,True
4,A. Diaz,True,True
...,...,...,...
2424,É. Bayen,False,True
2425,Élisa Wrembel,False,True
2426,Émilie Pecchi,False,True
2427,Éric Moreau,False,True


In [33]:
output_auteurs = f"auteurs_{LABO_CIBLE}_{ANNEE_CIBLE}.csv"
df_auteurs.to_csv(output_auteurs, index=False, encoding="utf-8-sig")
print(f"Fichier exporté : {output_auteurs}")

Fichier exporté : auteurs_LIS_2022.csv


---
## Analyse des auteurs par ORCID

Comparaison HAL ↔ OpenAlex basée sur les identifiants ORCID plutôt que sur les noms, ce qui évite les problèmes de découpage prénom / nom.

- Les auteurs **avec ORCID** sont vérifiés directement via l'API OpenAlex `/authors`.
- Les auteurs **sans ORCID** dans HAL restent comparés par nom normalisé (via la liste `authors_A` de la Solution A).

In [ ]:
# ── Extraction des ORCIDs HAL pour LABO_CIBLE / ANNEE_CIBLE ──────────────────
mask = (df["laboratoire"] == LABO_CIBLE) & (df["annee"] == ANNEE_CIBLE)

orcid_to_name = {}    # {orcid_nu -> nom HAL}
name_no_orcid = {}    # {nom_normalisé -> nom HAL} pour auteurs sans ORCID

for _, row in df[mask].iterrows():
    names  = [n.strip() for n in (row["auteurs"] or "").split("; ") if n.strip()]
    orcids = [_bare_orcid(o) for o in (row["orcids"] or "").split("; ")]
    orcids += [""] * max(0, len(names) - len(orcids))

    for name, orcid in zip(names, orcids):
        if orcid:
            orcid_to_name[orcid] = name
        else:
            name_no_orcid[_norm_title(name)] = name

print(f"Auteurs HAL {LABO_CIBLE} {ANNEE_CIBLE}")
print(f"  Avec ORCID : {len(orcid_to_name)}")
print(f"  Sans ORCID : {len(name_no_orcid)}")
print()
print("Auteurs avec ORCID :")
for orcid, name in sorted(orcid_to_name.items(), key=lambda x: x[1].split()[-1]):
    print(f"  {name:40s}  {orcid}")

In [35]:
# ── Recherche des ORCIDs dans OpenAlex (par lots) ────────────────────────────
def norm_orcid(orcid: str) -> str:
    """Normalise l'ORCID au format URI attendu par OpenAlex."""
    orcid = orcid.strip()
    if not orcid.startswith("https://"):
        orcid = f"https://orcid.org/{orcid}"
    return orcid


orcid_list   = list(orcid_to_name.keys())
found_orcids = set()   # ORCIDs (bruts) trouvés dans OpenAlex

for i in range(0, len(orcid_list), OA_BATCH_SIZE):
    batch      = [norm_orcid(o) for o in orcid_list[i : i + OA_BATCH_SIZE]]
    filter_val = "|".join(batch)
    params = {
        "filter":   f"orcid:{filter_val}",
        "select":   "id,orcid,display_name",
        "per_page": OA_BATCH_SIZE,
        "mailto":   OPENALEX_MAILTO,
    }
    try:
        resp = requests.get("https://api.openalex.org/authors",
                            params=params, timeout=30)
        resp.raise_for_status()
        for author in resp.json().get("results", []):
            raw = (author.get("orcid") or "").replace("https://orcid.org/", "")
            if raw:
                found_orcids.add(raw)
    except Exception as e:
        print(f"  ⚠ Erreur lot {i} : {e}")
    time.sleep(OA_SLEEP_SEC)

print(f"ORCIDs trouvés dans OpenAlex : {len(found_orcids)} / {len(orcid_list)}")
print(f"ORCIDs HAL non trouvés       : {len(orcid_list) - len(found_orcids)}")

ORCIDs trouvés dans OpenAlex : 116 / 122
ORCIDs HAL non trouvés       : 6


In [ ]:
# ── Dataframe de comparaison ORCID ───────────────────────────────────────────
# found_orcids et orcid_to_name utilisent tous les deux le format nu (0000-xxxx)
rows = []

# Auteurs AVEC ORCID dans HAL
for orcid, name in orcid_to_name.items():
    orcid_bare = _bare_orcid(orcid)   # sécurité : normalise même si déjà nu
    rows.append({
        "auteur":        name,
        "orcid":         orcid_bare,
        "dans_HAL":      True,
        "dans_OpenAlex": orcid_bare in found_orcids,
        "methode":       "orcid",
    })

# Auteurs SANS ORCID dans HAL : fallback comparaison par nom (via authors_A)
for key, name in name_no_orcid.items():
    rows.append({
        "auteur":        name,
        "orcid":         "",
        "dans_HAL":      True,
        "dans_OpenAlex": key in authors_A,
        "methode":       "nom",
    })

df_auteurs_orcid = (
    pd.DataFrame(rows)
    .sort_values(["methode", "dans_OpenAlex", "auteur"],
                 ascending=[True, False, True])
    .reset_index(drop=True)
)

mask_orcid = df_auteurs_orcid["methode"] == "orcid"
mask_nom   = df_auteurs_orcid["methode"] == "nom"
print(f"Auteurs {LABO_CIBLE} {ANNEE_CIBLE} — comparaison par ORCID")
print(f"  Avec ORCID → trouvés dans OpenAlex     : "
      f"{(mask_orcid &  df_auteurs_orcid['dans_OpenAlex']).sum()}")
print(f"  Avec ORCID → absents d'OpenAlex        : "
      f"{(mask_orcid & ~df_auteurs_orcid['dans_OpenAlex']).sum()}")
print(f"  Sans ORCID → trouvés par nom           : "
      f"{(mask_nom   &  df_auteurs_orcid['dans_OpenAlex']).sum()}")
print(f"  Sans ORCID → absents d'OpenAlex        : "
      f"{(mask_nom   & ~df_auteurs_orcid['dans_OpenAlex']).sum()}")
print(f"  Total                                  : {len(df_auteurs_orcid)}")
print()
# Cohérence : doit correspondre à la cellule précédente
print(f"  Vérif. found_orcids : {len(found_orcids)} ORCID dans OpenAlex "
      f"(attendu : {(mask_orcid & df_auteurs_orcid['dans_OpenAlex']).sum()})")

df_auteurs_orcid

In [37]:
output_orcid = f"auteurs_orcid_{LABO_CIBLE}_{ANNEE_CIBLE}.csv"
df_auteurs_orcid.to_csv(output_orcid, index=False, encoding="utf-8-sig")
print(f"Fichier exporté : {output_orcid}")

Fichier exporté : auteurs_orcid_LIS_2022.csv


---
## Origine des publications HAL absentes d'OpenAlex

Analyse des éditeurs, revues et conférences pour les publications non trouvées dans OpenAlex (sur l'ensemble de `df_unique`).

In [39]:
not_in_oa = df_unique[~df_unique["in_openalex"]].copy()
TOP_N = 40

print(f"Publications HAL non trouvées dans OpenAlex : {len(not_in_oa)} "
      f"/ {len(df_unique)} ({100*len(not_in_oa)/len(df_unique):.1f}%)\n")


def _scalar(series: pd.Series) -> pd.Series:
    """Extrait le premier élément si la valeur est une liste (retour HAL variable)."""
    return series.apply(lambda x: x[0] if isinstance(x, list) else x)


def top_table(col: str, n: int = TOP_N) -> pd.DataFrame:
    """
    Pour chaque valeur de `col`, affiche :
      - n_total      : total dans df_unique
      - n_absentes   : non trouvées dans OpenAlex
      - pct_absentes : n_absentes / n_total × 100
    Trié par n_absentes décroissant, limité aux n premières lignes.
    """
    total   = _scalar(df_unique[col]).value_counts(dropna=False).rename("n_total")
    missing = _scalar(not_in_oa[col]).value_counts(dropna=False).rename("n_absentes")
    return (
        pd.concat([total, missing], axis=1)
        .fillna(0)
        .astype({"n_total": int, "n_absentes": int})
        .assign(pct_absentes=lambda d: (d["n_absentes"] / d["n_total"] * 100).round(1))
        .sort_values("n_absentes", ascending=False)
        .head(n)
    )


print(f"── Top {TOP_N} éditeurs ──────────────────────────────")
display(top_table("editeur"))

print(f"\n── Top {TOP_N} revues ───────────────────────────────")
display(top_table("journal_revue"))

print(f"\n── Top {TOP_N} conférences ──────────────────────────")
display(top_table("conference"))

Publications HAL non trouvées dans OpenAlex : 57730 / 120087 (48.1%)

── Top 40 éditeurs ──────────────────────────────


,n_total,n_absentes,pct_absentes
editeur,,,
NaN,99893,53190,53.2
IEEE,6385,733,11.5
Springer,2232,463,20.7
ATALA,185,185,100.0
ACM,1570,109,6.9
ATALA & AFPC,62,62,100.0
Springer International Publishing,1616,58,3.6
CEUR-WS.org,58,58,100.0
Elsevier,268,56,20.9



── Top 40 revues ───────────────────────────────


,n_total,n_absentes,pct_absentes
journal_revue,,,
NaN,81136,53824,66.3
Interstices,105,104,99.0
Journal of Machine Learning Research,85,83,97.6
Bulletin de l'Association Française pour l'Intelligence Artificielle,63,63,100.0
Transactions on Machine Learning Research Journal,53,46,86.8
Docs en stock : dans les coulisses de la démocratie universitaire,43,43,100.0
Revue TAL : traitement automatique des langues,39,39,100.0
The Conversation France,38,34,89.5
ERCIM News,34,34,100.0



── Top 40 conférences ──────────────────────────


,n_total,n_absentes,pct_absentes
conference,,,
NaN,60951,22746,37.3
23ème congrès annuel de la Société Française de Recherche Opérationnelle et d'Aide à la Décision,69,69,100.0
GRETSI,57,57,100.0
CFA 2025 - 17e Congrès Français d'Acoustique,52,52,100.0
GRETSI 2023 - XXIXème Colloque Francophone de Traitement du Signal et des Images,50,50,100.0
20e Conférence en Recherche d’Information et Applications (CORIA) 32ème Conférence sur le Traitement Automatique des Langues Naturelles (TALN) 27ème Rencontre des Étudiants Chercheurs en Informatique pour le Traitement Automatique des Langues (RECITAL) Les 18e Rencontres Jeunes Chercheurs en RI (RJCRI),37,37,100.0
Plate-Forme Intelligence Artificielle,36,36,100.0
International Conference on Machine Learning,36,36,100.0
35èmes Journées d'Études sur la Parole (JEP 2024) 31ème Conférence sur le Traitement Automatique des Langues Naturelles (TALN 2024) 26ème Rencontre des Étudiants Chercheurs en Informatique pour le Traitement Automatique des Langues (RECITAL 2024),29,29,100.0
